In [ ]:
# This notebook is designed to run inside Google Colab. The automated commentary dependS on Gemini for text generation.

# Note that the packages listed below are normally already installed in Google Colab, so installing them again would be redundant.
# !pip install requests pandas numpy matplotlib scikit-learn fpdf2

# As mentioned in the README, a Coingecko API key is required to run this notebook with live data. Obtaining this key is free and done in less than 2 minutes in these steps.
# 1. Get a demo API key for free after sign-up, on: https://www.coingecko.com/en/api
# 2. Paste the key below
# 3. "Run all" to run everything!

# Important: Avoid spamming the 'Run all' button because CoinGecko's and DefiLlama's public APIs have rate limits to prevent abuse.
# Running the notebook multiple times in quick succession will trigger timeouts causing fetches to fail again.
# In such a case just wait 1-2 minutes between runs and/or run cells one by one. If you hit a 429 error pause for 5-10 minutes before retrying.

API_KEY = "your_key"       # paste yours here

In [ ]:
!pip install fpdf2

import requests
import pandas as pd
import numpy as np
import time
import os
import matplotlib.pyplot as plt
import urllib.request

from fpdf import FPDF, XPos, YPos
from datetime import datetime
from google.colab import ai

OUTPUT_FOLDER = "defi_report_assets"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
# Configuration

PROTOCOLS = {
    "Pendle": {
        "coingecko_id": "pendle",
        "defillama_slug": "pendle",
        "revenue_accrues_to_holder": True    # vePENDLE lockers receive the protocol fees
    },
    "Ethena": {
        "coingecko_id": "ethena",
        "defillama_slug": "ethena",
        "revenue_accrues_to_holder": False   # yield accrues to sUSDe stakers, not to ENA holders
    },
    "GMX": {
        "coingecko_id": "gmx",
        "defillama_slug": "gmx",
        "revenue_accrues_to_holder": True    # GMX stakers receive esGMX plus ETH from fee distributions
    },
    "Lido": {
        "coingecko_id": "lido-dao",
        "defillama_slug": "lido",
        "revenue_accrues_to_holder": False   # revenue flows to the DAO treasury, LDO is governance only
    },
    "Curve Finance": {
        "coingecko_id": "curve-dao-token",
        "defillama_slug": "curve-finance",
        "revenue_accrues_to_holder": True    # veCRV lockers receive 50% of the trading fees plus bribes
    },
    "Sky": {
        "coingecko_id": "sky",
        "defillama_slug": "sky",
        "revenue_accrues_to_holder": True    # SKY stakers receive the DSR spread and RWA yield
    }
}


COINGECKO_CACHE = {}   # a cache per run/protocol so we don't call the same one again

COLORS = {
    "Lido": "#00A3FF",
    "Pendle": "#FF6B00",
    "Curve Finance": "#FF5C5C",
    "GMX": "#4C6FFF",
    "Ethena": "#0A8F8C",
    "Sky": "#1A73E8"
}

In [ ]:
# Data fetching

def fetch_coingecko_data(coin_id, api_key=None):
    if coin_id in COINGECKO_CACHE:
        print(f"CoinGecko {coin_id} from cache")
        return COINGECKO_CACHE[coin_id]

    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}"
    headers = {}
    if api_key:
        headers["x-cg-demo-api-key"] = api_key

    for attempt in range(5):
        r = requests.get(url, headers=headers)
        print(f"CoinGecko {coin_id} - Status: {r.status_code}")

        if r.status_code == 200:
            data = r.json()
            result = {
                "price": data["market_data"]["current_price"]["usd"],
                "market_cap": data["market_data"]["market_cap"]["usd"],
                "circulating_supply": data["market_data"]["circulating_supply"],
                "total_supply": data["market_data"]["total_supply"],
                "max_supply": data["market_data"]["max_supply"]
            }
            COINGECKO_CACHE[coin_id] = result
            return result

        elif r.status_code == 429:
            wait = (2 ** attempt) * 5
            print(f"Rate limit hit. Waiting {wait}s...")
            time.sleep(wait)
        else:
            print(f"CoinGecko error {r.status_code}: {r.text[:200]}")
            time.sleep(2 ** attempt)

    raise Exception(f"Failed to fetch CoinGecko data for {coin_id} after retries")



def fetch_tvl_history(protocol_slug):
    url = f"https://api.llama.fi/protocol/{protocol_slug}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"
    }

    for attempt in range(3):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            print(f"DefiLlama {protocol_slug} - Status: {r.status_code}")

            # If it's not JSON (HTML error page) print the first 300 characters for diagnosis
            if not r.text.strip().startswith(("{", "[")):
                print("Non-JSON response received. First 300 chars:\n", r.text[:300])

            r.raise_for_status()
            data = r.json()

            # DefiLlama response often has top-level "tvl" (aggregate) but sometimes only per-chain. This handles both safely.
            if "tvl" in data and isinstance(data["tvl"], list) and len(data["tvl"]) > 0:
                tvl_list = data["tvl"]
            else:
                raise KeyError(f"No 'tvl' array found in response for {protocol_slug}")

            tvl_data = pd.DataFrame(tvl_list)
            tvl_data["date"] = pd.to_datetime(tvl_data["date"], unit="s")
            tvl_data = tvl_data[["date", "totalLiquidityUSD"]].rename(columns={"totalLiquidityUSD": "tvl"})
            tvl_data = tvl_data.dropna()

            last_date = tvl_data["date"].iloc[-1] if not tvl_data.empty else None

            return tvl_data, last_date, data

        except Exception as e:
            print(f"DefiLlama attempt {attempt+1} for {protocol_slug} failed: {e}")
            time.sleep(2 ** attempt)

    raise Exception(f"Failed to fetch TVL for {protocol_slug} after retries")

In [ ]:
#  Calculations of the metrics

def calculate_tvl_growth(tvl_df):
    tvl_df = tvl_df.dropna().sort_values("date")
    if len(tvl_df) < 30:
        return 0.0

    end_date = tvl_df["date"].max()
    recent = tvl_df[tvl_df["date"] >= (end_date - pd.Timedelta(days=365))]
    days = (recent["date"].iloc[-1] - recent["date"].iloc[0]).days

    if days <= 0:
        return 0.0

    start_tvl = recent["tvl"].iloc[0]
    end_tvl = recent["tvl"].iloc[-1]

    if start_tvl <= 0:
        return 0.0

    cagr = (end_tvl / start_tvl) ** (365 / days) - 1
    return cagr



def calculate_future_dilution_potential(circ, total, max_s):
    cap = max_s if max_s is not None else total
    if cap is None or cap == 0:
        return 0.0
    return max(0.0, (cap - circ) / cap)



def emission_pressure(supply_overhang, tvl_growth):
    """
    This is a heuristic metric that combines supply overhang with TVL momentum. It is used to compare protocols easier within this group but
    it does not come from the official financial theory.
    A positive TVL growth can absorb some unlock pressure, while negative growth will amplify this pressure with a penalty.
    but it should not be interpreted as a standard financial metric.
    """

    if tvl_growth > 0:
        raw = supply_overhang / max(tvl_growth, 0.05)       # minimum 5% growth is used to prevent a division with values too close to 0
        return min(raw, 10.0)                               # a cap at 10 to avoid extremes
    else:
        penalty = abs(tvl_growth) * 2.0                     # it depends on how negative, for example a -50% will get a +1.0 extra point
        return min(supply_overhang * 2.0 + penalty, 10.0)



# The main analysis

results = {}

for name, config in PROTOCOLS.items():
    print(f"\n  Processing {name}  ")
    try:
        cg = fetch_coingecko_data(config["coingecko_id"], API_KEY)
        tvl_df, last_tvl_date, full_data = fetch_tvl_history(config["defillama_slug"])
        tvl_growth = calculate_tvl_growth(tvl_df)
        supply_overhang = calculate_future_dilution_potential(cg["circulating_supply"], cg["total_supply"], cg["max_supply"])

        # Fetch revenue using DefiLlama and annualize it
        annual_estimated_revenue = np.nan
        revenue_url = f"https://api.llama.fi/summary/fees/{config['defillama_slug']}?dataType=dailyRevenue"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"
        }
        try:
            r = requests.get(revenue_url, headers=headers, timeout=30)
            r.raise_for_status()
            revenue_data = r.json()
            chart = revenue_data.get("totalDataChart", [])
            if chart:
                revenue_df = pd.DataFrame(chart, columns=["date", "dailyRevenue"])
                revenue_df["date"] = pd.to_datetime(revenue_df["date"], unit="s")
                recent_revenue = revenue_df.tail(30)
                if not recent_revenue.empty:
                    daily_avg_revenue = recent_revenue["dailyRevenue"].mean()
                    annual_estimated_revenue = daily_avg_revenue * 365
        except Exception as e:
            print(f"Revenue fetch failed for {name}: {e}")

        if not np.isnan(annual_estimated_revenue) and annual_estimated_revenue != 0:
            ps_ratio = cg["market_cap"] / annual_estimated_revenue
            supply_for_fdv = cg["max_supply"] or cg["total_supply"]
            fdv = cg["price"] * supply_for_fdv if supply_for_fdv else cg["market_cap"]
            fdv_ps = fdv / annual_estimated_revenue
            print(f"   Annual Revenue (30d avg):  ${annual_estimated_revenue:,.0f}")
        else:
            ps_ratio = np.nan
            fdv_ps = np.nan
            print(f"   Annual Revenue (30d avg):  N/A")

        base_metrics = {
            "market_cap": cg["market_cap"],
            "tvl": tvl_df["tvl"].iloc[-1],
            "supply_overhang": supply_overhang,
            "tvl_growth": tvl_growth,
            "emission_pressure": emission_pressure(supply_overhang, tvl_growth),
            "ps_ratio": ps_ratio,
            "fdv_ps": fdv_ps,
            "last_tvl_date": last_tvl_date.strftime("%Y-%m-%d") if last_tvl_date else "Unknown"
        }
        results[name] = {"base": base_metrics}

        print(f"   TVL Growth (annualized):   {tvl_growth:+.2%}")
        print(f"   Future Dilution Potential: {supply_overhang:.4f}  (undistributed supply / max supply)")
        print(f"   Emission Pressure:         {base_metrics['emission_pressure']:.2f}")
        print(f"   P/S Ratio:                 {base_metrics['ps_ratio']:.1f}")
        print(f"   FDV/Revenue:               {base_metrics['fdv_ps']:.1f}")

    except requests.exceptions.HTTPError as e:
        print(f"Error fetching data for {name}: {e}")
        print(f"Response content: {e.response.text}")
    except Exception as e:
        print(f"An unexpected error occurred for {name}: {e}")
    finally:
        time.sleep(10)



# Normalization for the radar charts

def min_max_normalize(values):
    valid = [v for v in values if not np.isnan(v)]
    if not valid:
        return [np.nan] * len(values)
    min_val = min(valid)
    max_val = max(valid)
    if max_val - min_val == 0:
        return [0.5 if not np.isnan(v) else np.nan for v in values]
    return [(v - min_val) / (max_val - min_val) if not np.isnan(v) else np.nan for v in values]

protocol_names = list(results.keys())

metrics = {
    "Future Dilution Potential": [results[p]["base"]["supply_overhang"] for p in protocol_names],
    "Emission Pressure": [results[p]["base"]["emission_pressure"] for p in protocol_names],
    "Valuation Risk (P/S)": [results[p]["base"].get("ps_ratio", np.nan) for p in protocol_names],
    "TVL Instability": [max(0, -results[p]["base"]["tvl_growth"]) for p in protocol_names],
    "FDV / Revenue": [results[p]["base"].get("fdv_ps", np.nan) for p in protocol_names]
}

# Apply log scaling to outlier-prone metrics for better normalization without hard caps
for key in ["Valuation Risk (P/S)", "FDV / Revenue", "Emission Pressure"]:
    metrics[key] = [np.log1p(max(v, 0)) if not np.isnan(v) else np.nan for v in metrics[key]]  # log1p for positive values, handles 0

normalized_metrics = {}
for metric_name, values in metrics.items():
    normalized_metrics[metric_name] = min_max_normalize(values)



# Composite score with the  normalized metrics

for idx, p in enumerate(protocol_names):
    base = results[p]["base"]
    has_revenue = not np.isnan(base.get("ps_ratio", np.nan))
    accrues = PROTOCOLS[p].get("revenue_accrues_to_holder", True)

    if has_revenue and accrues:
        # P/S is a signal of holder value
        base["composite_score"] = (
            0.25 * normalized_metrics["Future Dilution Potential"][idx] +
            0.20 * normalized_metrics["Emission Pressure"][idx] +
            0.20 * normalized_metrics["Valuation Risk (P/S)"][idx] +
            0.20 * normalized_metrics["TVL Instability"][idx] +
            0.15 * normalized_metrics["FDV / Revenue"][idx]
        )
    elif has_revenue and not accrues:
        # The case where P/S exists at protocol level but doesn't go to the token holders
        # We halve P/S weight and add it to the first two that matter more
        base["composite_score"] = (
            0.30 * normalized_metrics["Future Dilution Potential"][idx] +
            0.25 * normalized_metrics["Emission Pressure"][idx] +
            0.10 * normalized_metrics["Valuation Risk (P/S)"][idx] +
            0.20 * normalized_metrics["TVL Instability"][idx] +
            0.15 * normalized_metrics["FDV / Revenue"][idx]
        )
    else:
        # if no revenue data we spread the weight across the other 3 metrics
        base["composite_score"] = (
            0.40 * normalized_metrics["Future Dilution Potential"][idx] +
            0.35 * normalized_metrics["Emission Pressure"][idx] +
            0.25 * normalized_metrics["TVL Instability"][idx]
        )



# Radar chart function
colors = [COLORS[p] for p in protocol_names]

def plot_radar(protocol_index, color):
    labels = list(normalized_metrics.keys())
    values = [normalized_metrics[label][protocol_index] for label in labels]
    values += values[:1]

    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.plot(angles, values, linewidth=2.5, color=color, label=protocol_names[protocol_index])
    ax.fill(angles, values, alpha=0.25, color=color)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_title(f"{protocol_names[protocol_index]} Risk Profile\n(Larger Area = Higher Risk)", pad=20, fontsize=13)
    ax.set_ylim(0, 1)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

for i in range(len(protocol_names)):
    plot_radar(i, colors[i])
    plt.savefig(f"{OUTPUT_FOLDER}/{protocol_names[i]}_radar.png", bbox_inches='tight', dpi=200, facecolor='white')
    plt.close()



# Projection chart visualization
inflations = [results[p]["base"]["supply_overhang"] for p in protocol_names]
pressures = [results[p]["base"]["emission_pressure"] for p in protocol_names]


def dilution_projection(circ, total, max_s, years=3):
    cap = max_s if max_s is not None else total
    if cap is None or cap == 0:
        return [circ] * (years + 1)          # uncapped flat line

    remaining = max(0.0, cap - circ)
    if remaining == 0:
        return [circ] * (years + 1)

    annual_unlock = remaining / years
    projections = []
    current = circ
    for _ in range(years + 1):
        projections.append(current)
        current += annual_unlock
    return projections

years = list(range(0, 4))



# Build the supply projection data before the charts
supply_data = {}
for name, config in PROTOCOLS.items():
    cg = fetch_coingecko_data(config["coingecko_id"], API_KEY)
    projection = dilution_projection(cg["circulating_supply"], cg["total_supply"], cg["max_supply"])
    supply_data[name] = [p / 1e6 for p in projection]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for name, proj in supply_data.items():
    ax1.plot(years, proj, label=name, marker='o', linewidth=2, color=COLORS[name])
    ax2.plot(years, proj, label=name, marker='o', linewidth=2, color=COLORS[name])

ax1.set_title("Linear Scale")
ax1.set_xlabel("Years from now")
ax1.set_ylabel("Projected Circulating Supply (millions)")
ax1.set_xticks([0, 1, 2, 3])
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_yscale("log")
ax2.set_title("Log Scale (separates smaller protocols)")
ax2.set_xlabel("Years from now")
ax2.set_ylabel("Projected Circulating Supply (millions, log scale)")
ax2.set_xticks([0, 1, 2, 3])
ax2.legend()
ax2.grid(True, alpha=0.3, which="both")

fig.suptitle("3-Year Circulating Supply Projection (in millions)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_FOLDER}/supply_projection_millions.png", bbox_inches='tight', dpi=200, facecolor='white')
plt.close()




# Relative dillution chart

plt.figure(figsize=(10, 6))

for name, config in PROTOCOLS.items():
    cg = fetch_coingecko_data(config["coingecko_id"], API_KEY)
    projection = dilution_projection(
        cg["circulating_supply"],
        cg["total_supply"],
        cg["max_supply"])
    if projection:
        years = list(range(0, 4))
        # Normalize to % of max supply
        max_cap = cg["max_supply"] if cg["max_supply"] is not None else cg["total_supply"]
        if max_cap and max_cap > 0:
            rel = [ (p / max_cap) * 100 for p in projection ]
            plt.plot(years, rel, label=name, marker='o', linewidth=2.5, color=COLORS[name])

plt.title("Relative Dilution – % of Max Supply Unlocked (3-Year Projection)")
plt.xlabel("Years from now")
plt.ylabel("Circulating Supply (% of Max Cap)")
plt.xticks([0, 1, 2, 3])
plt.ylim(0, 105)
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig(f"{OUTPUT_FOLDER}/relative_dilution.png", bbox_inches='tight', dpi=200, facecolor='white')
plt.close()




# Composite score ranking

plt.figure(figsize=(8, 5))

# Sort worst-to-best (descending order) so the chart reads as a clear ranking left to right
sorted_protocols = sorted(protocol_names, key=lambda p: results[p]["base"]["composite_score"], reverse=True)
scores = [results[p]["base"]["composite_score"] for p in sorted_protocols]
colors_bar = [COLORS[p] for p in sorted_protocols]

plt.bar(sorted_protocols, scores, color=colors_bar)
plt.title("Sustainability Composite Score (LOWER = Better)", fontsize=14)
plt.ylabel("Composite Risk Score")
plt.grid(axis='y', alpha=0.3)

for i, v in enumerate(scores):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center', fontsize=11)

plt.savefig(f"{OUTPUT_FOLDER}/composite_score_ranking.png", bbox_inches='tight', dpi=200, facecolor='white')
plt.close()

print(f"\nTop Performer: {sorted_protocols[scores.index(min(scores))]} exhibits the lowest composite risk score.")

In [ ]:
# PDF commentary text

protocol_names = list(PROTOCOLS.keys())

# Winner
winner = min(protocol_names, key=lambda p: results[p]["base"]["composite_score"])
win_score = results[winner]["base"]["composite_score"]

# TVL in billions for descriptions
tvl_b = {p: results[p]["base"]["tvl"] / 1_000_000_000 for p in protocol_names}

# 3-year dilution %
def get_dilution_pct(name):
    cg = fetch_coingecko_data(PROTOCOLS[name]["coingecko_id"], API_KEY)
    proj = dilution_projection(cg["circulating_supply"], cg["total_supply"], cg["max_supply"], years=3)
    start = proj[0]
    end = proj[-1]
    if start > 0:
        return round(((end - start) / start) * 100)
    return 0

dil = {name: get_dilution_pct(name) for name in protocol_names}

dynamic_summary = (
    "This is an assessment of future sustainability for a group of DeFi protocols that uses live data from CoinGecko and DefiLlama. "
    "It calculates five critical metrics: annualized TVL growth, future dilution potential from supply schedules, "
    "emission pressure relative to growth, P/S ratio based on real fee revenue and FDV/Revenue. "
    "They are then combined into a composite risk score, where lower = more sustainable. "
    "Emission pressure is a heuristic metric created here after trial and error. "
    "It assesses how effectively a protocol's growth may absorb or increase the dilution created by new tokens that enter the circulation. "
    "It has two sides based on direction and both are capped at 10 to avoid extreme values. "
    "When TVL growth is positive it is calculated with the simpler formula supply_overhang / TVL_growth. "
    "When TVL growth is negative the formula changes to (supply_overhang x 2) + (|decline| x 2). "
    "The composite scores are relative to this peer group and they will change with different protocols. "
    "It also has two different weights based on whether token revenue goes to token holders or not. "
    "If it does, the weights are: Future Dilution Potential 25%, Emission Pressure 20%, Valuation Risk (P/S) 20%, "
    "TVL Instability 20%, FDV/Revenue 15%. "
    "If it doesn't they become: Future Dilution Potential 30%, Emission Pressure 25%, Valuation Risk (P/S) 10%, "
    "TVL Instability 20%, FDV/Revenue 15%. "
    "This happenes because market cap / protocol revenue isn't a holder value signal in governance only tokens and "
    "more attention can be placed on the dilution metrics."
)

pendle_text = (
    f"Pendle has been the leader in yield-tokenization. This is about separating a yield bearing asset into two components, principal and yield, "
    f"so traders can take fixed or leveraged yield positions. At ${tvl_b['Pendle']:.1f}B TVL it is the market maker for yield trading on chain. "
    f"The {results['Pendle']['base']['supply_overhang']:.4f} supply overhang reflects ongoing liquidity incentives, "
    f"though these are offset a bit by fee revenue going to sPENDLE stakers via buybacks, as vePENDLE was replaced on January 2026. "
    f"TVL peaked near $7B in mid 2024 due to programs for point farming by Eigenlayer and Ethena. "
    f"The current situation shows steady demand even after these incentives were completed."
)

ethena_text = (
    f"Ethena is the pioneer of synthetic dollar, USDe and delta-neutral yield with ${tvl_b['Ethena']:.1f}B+ TVL. "
    f"This was included as an important stablecoin innovation of 2025-2026. "
    f"With {results['Ethena']['base']['supply_overhang']:.4f} supply overhang ratio and {results['Ethena']['base']['tvl_growth']:+.2%} TVL growth, "
    f"it shows good funding mechanics that create a sustainable yield."
)

gmx_text = (
    f"GMX has got the token-holder alignment right early. Stakers receive actual ETH fee distributions and not just new, inflationary rewards. "
    f"V1 sent 30% to GMX stakers, V2 revised the split across GLP/GM pools. "
    f"Controlled inflation and {results['GMX']['base']['tvl_growth']:+.2%} TVL growth make it a useful reference point against the other, emission heavy protocols in this group. "
    f"It shows what revenue-aligned tokenomics can actually look like."
)

lido_text = (
    f"Lido is the biggest liquid staking protocol by a wide margin, with TVL of ${tvl_b['Lido']:.1f}B+. It is deeply embedded in Ethereum's staking infrastructure. "
    f"LDO is a governance token, protocol revenue flows to the DAO treasury and not directly to the LDO holders. This can make the P/S figure less useful for token investors than it looks. "
    f"The supply overhang at {results['Lido']['base']['supply_overhang']:.4f} is relatively contained, and a TVL growth of {results['Lido']['base']['tvl_growth']:+.2%} reflects a mostly stable base with real utility and token value."
)

curve_text = (
    f"In Curve's veCRV model lockers receive 50% of trading fees plus bribes from protocols, that compete for liquidity direction. It also appears to be one of the most copied tokenomics designs in DeFi. "
    f"The {results['Curve Finance']['base']['supply_overhang']:.4f} supply overhang is among the highest in this group and a {results['Curve Finance']['base']['tvl_growth']:+.2%} TVL growth makes that harder to absorb, being too focused on emissions. "
    f"According to the Curve whitepaper CRV emissions follow an exponential decay curve, so the dilution figures in this report are a rough upper bound rather than a precise forecast."
)

sky_text = (
    f"Sky (formerly MakerDAO) is the oldest protocol in this group, the original decentralized stablecoin issuer. The rebrand from MKR at a 1:24,000 conversion ratio inflates the raw supply figures of the dilution metrics. "
    f"At a TVL of ${tvl_b['Sky']:.1f}B, a {results['Sky']['base']['tvl_growth']:+.2%} TVL growth and only {results['Sky']['base']['supply_overhang']:.4f} future dilution potential, "
    f"it is proven to be the more mature of this group. Buybacks and collateral interest have made it rewarding for long-term holders"
)

charts_text = (
    f"- The relative dilution chart shows what percentage of each token's maximum supply will unlock over the next 3 years, "
    f"revealing future selling pressure on holders. "
    f"Real vesting schedules may not be linear or they may be dynamic based on various factors, but we assume linear unlocks to avoid overcomplicating things.\n"
    f"- A 3-year circulating supply projection displays token growth, which allows for visual comparison of emission scale across protocols.\n"
    f"- A sustainability composite score ranking bar chart adds all five metrics into one score. "
    f"{winner} exhibits the lowest score at {win_score:.3f}, which indicates the strongest sustainability among the six."
)



# Sort protocols by their 3-year dilution % for the LLM to get their ranks
dil_sorted = sorted(dil.items(), key=lambda x: x[1])
dil_lowest = dil_sorted[:2]        # 2 best with the least dilution
dil_highest = dil_sorted[-2:]      # 2 worst with the most dilution
dil_middle = dil_sorted[2:-2]      # the remaining 2 in the middle


relative_dilution_text = (
    "3-year supply increase by protocol (% of circulating supply at start): "
    + ", ".join([f"{p} {v}%" for p, v in dil_sorted])
    + ". LOWER % = less dilution pressure on holders."
)

supply_projection_text = (
    f"Lowest dilution: {', '.join([f'{p} ({v}%)' for p, v in dil_lowest])}. "
    f"Mid-range: {', '.join([f'{p} ({v}%)' for p, v in dil_middle])}. "
    f"Highest dilution: {', '.join([f'{p} ({v}%)' for p, v in dil_highest])}."
)

composite_summary_metrics = " | ".join([f"{p}: {results[p]['base']['composite_score']:.3f}" for p in protocol_names])

In [ ]:
# PDF building

os.system("apt-get install -y -q fonts-dejavu")
pdf = FPDF(orientation='P', unit='mm', format='A4')

pdf.add_font("DejaVu", style="",   fname="/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf")
pdf.add_font("DejaVu", style="B",  fname="/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf")
pdf.add_font("DejaVu", style="I",  fname="/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf")
pdf.add_font("DejaVu", style="BI", fname="/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf")

pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()



# 1rst Page - Title & executive summary

# Main Title
pdf.set_font("DejaVu", "B", 18)
pdf.cell(0, 15, text="DeFi Token Sustainability Report", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
pdf.ln(8)

# Executive Summary
pdf.set_font("DejaVu", "B", 13)
pdf.cell(0, 8, text="Executive Summary", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
pdf.ln(4)

pdf.set_font("DejaVu", "", 10.8)
summary = dynamic_summary
pdf.multi_cell(0, 5.2, text=summary)
pdf.ln(7)

# Protocols Analyzed
pdf.set_font("DejaVu", "B", 12)
pdf.cell(0, 7, text="Protocols Analyzed & Why They Were Chosen", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.ln(3)

pdf.set_font("DejaVu", "", 10.5)

pdf.multi_cell(0, 5.1, text=pendle_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, text=ethena_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, text=gmx_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, text=lido_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, text=curve_text)
pdf.ln(3)
pdf.multi_cell(0, 5.1, text=sky_text)
pdf.ln(7)

# Visual Insights
pdf.set_font("DejaVu", "B", 12)
pdf.cell(0, 7, text="What the Final Three Charts Reveal", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
pdf.ln(3)

pdf.set_font("DejaVu", "", 10.5)
pdf.multi_cell(0, 5.2, text=charts_text)
pdf.ln(5)

pdf.set_font("DejaVu", "I", 9.5)
pdf.cell(0, 5, text=f"All data pulled {datetime.now().strftime('%B %Y')} One-click Colab. Built for the serious DeFi researcher", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')



# 2nd Page - Metrics glossary

pdf.add_page()

pdf.set_font("DejaVu", "B", 14)
pdf.cell(0, 10, text="Metric Definitions", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
pdf.ln(2)
pdf.set_font("DejaVu", "I", 9)
pdf.cell(0, 5, text="Reference guide to every metric shown in the protocol tables.", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
pdf.ln(7)

glossary_entries = [
    (
        "Market Cap",
        "Total market value of all tokens currently in circulation (price x circulating supply). "
        "A smaller figure means a less established market presence, while a larger one reflects broader adoption and liquidity."
    ),
    (
        "TVL (Total Value Locked)",
        "The total assets actively deposited and working inside the protocol. It's the most direct signal of real capital trust. "
        "Protocols with low TVL either haven't found product market fit yet or are losing user confidence."
    ),
    (
        "TVL Growth (annualized)",
        "Annualized CAGR (the compound annual growth rate) of TVL over the trailing 12 months. It measures whether the protocol is growing or shrinking, positive when capital is coming in or negative when it is leaving. "
        "Context matters a lot too, a protocol at $20B TVL growing 5% is a different story from a $200M protocol growing 5%."
    ),
    (
        "TVL Instability",
        "It is derived from TVL Growth with max(0, -TVL_growth). Protocols with positive TVL growth score 0, no instability penalty. "
        "Declining protocols are penalized in proportion to how fast they're contracting. "
        "This means a fast-growing protocol like Ethena will get a 0 here despite all the other risk factors."
    ),
    (
        "Future Dilution Potential",
        "This is the token supply that is not in circulation yet. Calculated by (max_supply - circ_supply) / max_supply. It ranges from 0 to 1. "
        "If it is near 0 it means that most supply is already circulating so the unlock pressure on holders is minimal. "
        "If it reaches near 1 it means that a large proportion of the supply is yet to be released, so the future sell pressure risk is greatly increased. "
        "We assume that all the stated remaining supply will do actually circulate. "
        "Protocols like CRV with continuous emission schedules can have a slower release rate than this figure."
    ),
    (
        "Emission Pressure",
        "This is a created, heuristic metric that combines the supply overhang (which is the undistributed supply as a fraction of max supply) "
        "with the TVL momentum. "
        "When TVL growth is positive the formula is overhang supply / TVL_growth. We enforce a minimum value of 5% to prevent low growth values from "
        "creating weird numbers due to the division. When the TVL has negative growth, the formula becomes (overhang x 2) + (|TVL decline| x 2). "
        "We put a cap of 10 to both cases. When this metric is low, it means that the unlock pressure is well-absorbed by the growth momentum. "
        "When it is high, it points to a significant supply waiting to enter the market against weak or declining protocol traction, which is bad. "
    ),
    (
        "P/S Ratio (Price to sales ratio)",
        "It is calculated by dividing market cap by the annualized protocol revenue (found by 30-day average daily revenue x 365). "
        "When it is low it means that the token is inexpensive relative to its current revenue generation and may be undervalued. "
        "If it is high, the token trades at a revenue premium, with the market expecting a greater future growth. "
        "The weight this ratio has in the composite score below changes a little for governance only tokens like ENA and LDO "
        "where the revenue flows to the protocol treasury rather than to the token holders. "
        "Only half the normal weight is placed on this ratio, as the dillution and emission metrics ar more important for these cases."
    ),
    (
        "FDV / Revenue",
        "This is the fully diluted valuation (price x max supply) divided by the annualized revenue. "
        "Compared to the P/S ratio this uses the full potential market cap instead. "
        "If FVD / Revenue is low it means that the protocol seems suitably priced even if all supply was in circulation today. "
        "If it is high, it means that the protocol is already priced for greatly increased future revenue growth, taking it for granted almost."
    ),
    (
        "Composite Score",
        "This is a weighted sum of all the five risk metrics, normalized for better comparisons in this peer group. "
        "The standard weights for the protocols where revenue accrues to token holders are the following: "
        "Future dilution potential 25%, Emission pressure 20%, P/S Ratio (valuation risk) 20%, "
        "TVL Instability 20% and FDV/Revenue 15%. "
        "For protocols where the revenue does not flow to token holders, governance onl tokens, the weights change a little: "
        "Future dilution potential 30%, Emission pressure 25%, P/S Ratio (valuation risk) 10%, "
        "TVL Instability 20% and FDV/Revenue 15%. As mentioned before, the P/S weight is halved for governance-only tokens. "
        "A lower score here means more sustainable tokenomics. Also, the scores are relative to this peer group and will shift "
        "if the protocols change."
    ),
]

for metric_name, explanation in glossary_entries:
    pdf.set_font("DejaVu", "B", 9.5)
    pdf.cell(0, 6, text=metric_name, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("DejaVu", "", 9)
    pdf.multi_cell(0, 4.8, text=explanation)
    pdf.ln(3.5)

In [ ]:
# Prompts for Gemini

def get_llm_commentary(title: str, metrics: str, context: str = '') -> str:

    prompt = f"""
    You are a professional DeFi token analyst writing for institutional investors.
    Write exactly three sentences (maximum 150 words total) about {title}'s long-term token sustainability based on its metrics.

    Rules:
    - Sentence 1: one positive or neutral observation using the strongest metric (TVL, inflation, or valuation).
    - Sentence 2: one risk observation using the weakest metric.
    - Sentence 3: wrap up with the Composite Score. Always use the exact phrase "Composite Score of X.XXX". Remember that a lower composite score means higher sustainability.

    State clearly whether this score places the protocol among the most sustainable (lowest scores) or least sustainable (highest scores) in the peer group.
    Do not repeat the rank label after already stating it, once is enough.

    Use only these exact metrics:
    Data: {metrics}
    Context: {context}

    One important thing on Emission Pressure is that it measures token inflation relative to TVL growth momentum,
    and not relative to revenue. Never describe it as exceeding or comparing to revenues.
    If Emission Pressure > 1, describe it as emissions outpacing TVL growth absorption capacity.

    Tone: Neutral, data-driven, pragmatic, no hype words like (robust, compelling, significant, notable, etc.).
    Stay concise. Never start sentence 2 with "However" and never start sentence 3 with "This results in" or "Consequently".
    Vary the transitions naturally. Start directly with analysis, no introduction.
    """

    try:
        response = ai.generate_text(prompt)
        return (response.strip())
    except Exception as e:
        return "AI commentary temporarily unavailable (Colab AI service issue)."



def get_chart_commentary(chart_context: str, insight_angle: str = '') -> str:
    prompt = f"""
    You are a professional DeFi token analyst writing chart commentary for an institutional investor report.
    Write a structured paragraph of exactly four sentences (150 words max) that will interpret this cross-protocol data:
    Data: {chart_context}
    Angle: {insight_angle}

    Rules:
    - Sentence 1: why this metric matters to token holders and what the chart is designed to reveal.
    - Sentence 2: the protocols at the top and bottom of the range with their exact figures.
    - Sentence 3: explain what it means for buying, holding or selling decisions, the practical investor implication of that spread.
    - Sentence 4: identify which protocols sit in the middle of the range and characterize their position relative to the extremes.

    Tone: neutral, data-driven, pragmatic. No introductions, be concise.
    Avoid filler phrases like "necessitating a more cautious approach", "requiring individual assessment" or any similar padding.
    Write like an analyst talking to a peer, not a report generator summarizing itself.
    Do NOT assign or mention any Composite Score, as this is a chart-level observation only. Do not use hype words like (robust, compelling, significant, notable, etc.).
    """

    try:
        response = ai.generate_text(prompt)
        return (response.strip())
    except Exception as e:
        return "Chart commentary temporarily unavailable."




# Protocol pages

for i, protocol in enumerate(protocol_names):
    pdf.add_page()
    base = results[protocol]["base"]

    # Protocol name
    pdf.set_font("DejaVu", "B", 14)
    pdf.cell(0, 12, text=protocol, new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
    pdf.ln(8)

    # Centered Table
    pdf.set_font("DejaVu", "B", 10)
    table_width = 125
    pdf.set_x((210 - table_width) / 2)

    pdf.cell(80, 8, "Metric", 1, align='C')
    pdf.cell(45, 8, "Value", 1, align='C')
    pdf.ln()

    pdf.set_font("DejaVu", "", 9.5)
    table_data = [
        ("Market Cap",                f"${base['market_cap']:,}"),
        ("TVL",                       f"${int(round(base['tvl'])):,}"),
        ("TVL as of",                 base.get("last_tvl_date", "Unknown")),
        ("Future Dilution Potential", f"{base['supply_overhang']:.4f}"),
        ("TVL Growth (ann.)",         f"{base['tvl_growth']:+.2%}"),
        ("Emission Pressure",         f"{base['emission_pressure']:.3f}"),
        ("P/S Ratio",                 "N/A" if np.isnan(base.get('ps_ratio', 0)) else f"{base['ps_ratio']:.2f}"),
        ("FDV / Revenue",             "N/A" if np.isnan(base.get('fdv_ps', 0)) else f"{base['fdv_ps']:.2f}"),
        ("Composite Score",           f"{base['composite_score']:.4f}"),
    ]

    for label, value in table_data:
        pdf.set_x((210 - table_width) / 2)
        pdf.cell(80, 7.5, label, 1)
        pdf.cell(45, 7.5, value, 1, align='R')
        pdf.ln()

    pdf.ln(8)



    # AI commentary

    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 8, text="AI Analyst Insight", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("DejaVu", "", 10.2)

    all_scores_sorted = sorted(protocol_names, key=lambda p: results[p]["base"]["composite_score"])
    rank = all_scores_sorted.index(protocol) + 1
    n_total = len(protocol_names)
    rank_label = (
        "most sustainable (rank 1)"        if rank == 1 else
        f"2nd most sustainable (rank {rank} of {n_total})"  if rank == 2 else
        f"middle of peer group (rank {rank} of {n_total})"  if rank <= n_total - 2 else
        f"2nd least sustainable (rank {rank} of {n_total})" if rank == n_total - 1 else
        "least sustainable (last place)"
    )

    metrics_str = (
        f"Market Cap: ${base['market_cap']:,.0f} | "
        f"TVL: ${base['tvl']:,.0f} | "
        f"Future Dilution Potential (0-1 scale. HIGHER = larger undistributed supply overhang = greater dilution risk for holders): {base['supply_overhang']:.4f} | "
        f"TVL Growth: {base['tvl_growth']:+.1%} | "
        f"Emission Pressure: {base['emission_pressure']:.3f} | "
        f"P/S: {'N/A' if np.isnan(base.get('ps_ratio', np.nan)) else f'{base['ps_ratio']:.2f}'} | "
        f"FDV/Revenue: {'N/A' if np.isnan(base.get('fdv_ps', np.nan)) else f'{base['fdv_ps']:.2f}'} | "
        f"Composite Score: {base['composite_score']:.4f} | "
        f"Peer group rank (LOWER composite = MORE sustainable): {rank} of {n_total} - this protocol is the {rank_label}. "
        f"You MUST use this exact rank description in sentence 3, do not reinterpret it."
    )

    protocol_context = {
        "Ethena": (
            "Compare briefly to the other protocols where relevant. "
            "Important: You must include in sentence 1 or 2 that ENA token holders have no direct claim on protocol revenue. "
            "Yield from the delta-neutral USDe position accrues to sUSDe stakers, not ENA holders. "
            "This explains the elevated P/S and FDV/Revenue multipliers despite real yield generation within the protocol."
            "Do not repeat the same point across two sentences."
        ),
        "Lido": (
            "Compare briefly to the other protocols where relevant. "
            "Important: You must include in sentence 1 or 2 that Lido's P/S ratio of {base['ps_ratio']:.2f} "
            "reflects protocol-level earnings that flow to the DAO treasury. "
            "LDO token holders have no direct revenue claim, unlike GMX stakers or veCRV lockers. "
            "This explains the elevated P/S and FDV/Revenue multipliers despite real yield generation within the protocol."
        ),
    }.get(protocol, "Compare briefly to the other protocols where relevant.")

    commentary = get_llm_commentary(
        protocol,
        metrics_str,
        context=protocol_context
    )

    pdf.multi_cell(0, 5.2, text=commentary)
    pdf.ln(6)



    # Radar Chart

    radar_file = f"{OUTPUT_FOLDER}/{protocol}_radar.png"
    if os.path.exists(radar_file):
        chart_width = 145
        pdf.set_x((210 - chart_width) / 2)
        pdf.image(radar_file, w=chart_width)

In [ ]:
# Final charts

# Relative Dilution page
pdf.add_page()
if os.path.exists(f"{OUTPUT_FOLDER}/relative_dilution.png"):
    pdf.set_font("DejaVu", "B", 14)
    pdf.cell(0, 12, text="Relative Dilution – % of Max Supply (3-Year)", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
    pdf.ln(10)
    pdf.set_x((210 - 170) / 2)
    pdf.image(f"{OUTPUT_FOLDER}/relative_dilution.png", w=170)



# Methodology footnote
    pdf.ln(6)
    pdf.set_font("DejaVu", "I", 8.5)
    pdf.set_text_color(100, 100, 100)
    pdf.multi_cell(0, 4.5, text=
        "Methodology note: Projections assume linear unlocks across 3 years. "
        "This is a simplification, as actual schedules vary significantly between different protocols. "
        "According to the Curve whitepapper CRV emissions follow an exponential decay curve so a very high figure is an overestimation of realistic dilution. "
        "Lido's LDO has no formal remaining unlock schedule, its figure reflects only the undistributed supply."
    )

    pdf.set_text_color(0, 0, 0)
    pdf.ln(4)
    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 8, text="AI Analyst Insight", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("DejaVu", "", 10.2)
    chart_comment = get_chart_commentary(
        relative_dilution_text,
        insight_angle="Explain the long-term selling pressure implication for token holders."
    )
    pdf.multi_cell(0, 5.2, text=chart_comment)



# Supply Projection page
pdf.add_page()
if os.path.exists(f"{OUTPUT_FOLDER}/supply_projection_millions.png"):
    pdf.set_font("DejaVu", "B", 14)
    pdf.cell(0, 12, text="3-Year Circulating Supply Projection (in millions)", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
    pdf.ln(10)
    pdf.set_x((210 - 170) / 2)
    pdf.image(f"{OUTPUT_FOLDER}/supply_projection_millions.png", w=170)



 # AI Insight
    pdf.ln(8)
    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 8, text="AI Analyst Insight", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("DejaVu", "", 10.2)
    chart_comment = get_chart_commentary(
        supply_projection_text,
        insight_angle="Explain how these emission rates affect token price sustainability over 3 years."
    )
    pdf.multi_cell(0, 5.2, text=chart_comment)



# Composite Score Ranking page
pdf.add_page()
if os.path.exists(f"{OUTPUT_FOLDER}/composite_score_ranking.png"):
    pdf.set_font("DejaVu", "B", 14)
    pdf.cell(0, 12, text="Sustainability Composite Score Ranking", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
    pdf.ln(15)
    pdf.set_x((210 - 170) / 2)
    pdf.image(f"{OUTPUT_FOLDER}/composite_score_ranking.png", w=170)



 # AI Insight
    pdf.ln(8)
    pdf.set_font("DejaVu", "B", 11)
    pdf.cell(0, 8, text="AI Analyst Insight", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("DejaVu", "", 10.2)
    chart_comment = get_chart_commentary(
        composite_summary_metrics,
        insight_angle="Explain what separates the top and bottom performers and what this means for investors. LOWER score = MORE sustainable."
    )
    pdf.multi_cell(0, 5.2, text=chart_comment)

pdf.output("defi_sustainability_report.pdf")
print("Final clean PDF generated!")